In [ ]:
# Pipeline parameters — overridden by job base_parameters when run via DAB.
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "dev")
dbutils.widgets.text("volume_name", "raw_files")
dbutils.widgets.text("bronze_write_mode", "overwrite")
dbutils.widgets.text("overwrite_schema", "true")
dbutils.widgets.text("null_high_severity_pct", "10")
dbutils.widgets.text("dq_sample_limit", "20")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
bronze_write_mode = dbutils.widgets.get("bronze_write_mode")
overwrite_schema = dbutils.widgets.get("overwrite_schema").lower() == "true"
null_high_severity_pct = float(dbutils.widgets.get("null_high_severity_pct"))
dq_sample_limit = int(dbutils.widgets.get("dq_sample_limit"))

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
print(f"catalog={catalog}  schema={schema}  volume_path={volume_path}")
print(f"bronze_write_mode={bronze_write_mode}  overwrite_schema={overwrite_schema}")

## Step 4 - Identify Duplicate Records Across Bronze Tables

In [ ]:
from actuarial_claim_pipeline.bronze import list_bronze_tables
from actuarial_claim_pipeline.data_quality import duplicate_summary, find_duplicates

bronze_tables = list_bronze_tables(spark, catalog, schema)
print(f"Scanning {len(bronze_tables)} bronze table(s) for duplicates …\n")

summary = []
total_dirty = 0

for table_name in bronze_tables:
    full_name = f"{catalog}.{schema}.{table_name}"
    df = spark.table(full_name)
    stats = duplicate_summary(df)
    dupe_groups = stats["dupe_groups"]
    rows_to_drop = stats["rows_to_drop"]
    status = "✅ Clean" if dupe_groups == 0 else "⚠️  Duplicates found"
    total_dirty += 1 if dupe_groups > 0 else 0

    summary.append({
        "table": full_name,
        "total_rows": stats["total_rows"],
        "dupe_groups": dupe_groups,
        "rows_to_drop": rows_to_drop,
        "dupes_df": find_duplicates(df) if dupe_groups > 0 else None,
    })

    print(f"  {full_name}")
    print(f"    Total rows         : {stats['total_rows']:,}")
    print(f"    Duplicate groups   : {dupe_groups:,}")
    print(f"    Extra rows to drop : {rows_to_drop:,}")
    print(f"    Status             : {status}")
    print()

print(f"{'─'*60}")
if total_dirty == 0:
    print("✅ All bronze tables are clean — no duplicates found.")
else:
    print(f"⚠️  {total_dirty} table(s) contain duplicate records.")
    print("   Run the deduplication cell to clean them.")
print(f"{'─'*60}")

for s in summary:
    if s["dupes_df"] is not None:
        print(f"\nSample duplicates — {s['table']}  (top by count):")
        display(s["dupes_df"].limit(dq_sample_limit))

In [ ]:
from actuarial_claim_pipeline.data_quality import (
    business_rule_violations,
    null_analysis,
    referential_integrity_violations,
)

claims = spark.table(f"{catalog}.{schema}.bronze_claims_bordereau")
events = spark.table(f"{catalog}.{schema}.bronze_cyclone_events")
premiums = spark.table(f"{catalog}.{schema}.bronze_premium_bordereau")
risk_zone = spark.table(f"{catalog}.{schema}.bronze_risk_zone_lookup")

issues = []

print("═" * 70)
print("  DATA QUALITY REPORT — Bronze Layer")
print("═" * 70)

print("\n📋 1. NULL / MISSING VALUE ANALYSIS")
print("─" * 70)

for tbl_name, df in [
    ("bronze_claims_bordereau", claims),
    ("bronze_cyclone_events", events),
    ("bronze_premium_bordereau", premiums),
    ("bronze_risk_zone_lookup", risk_zone),
]:
    nulls = null_analysis(df, high_threshold_pct=null_high_severity_pct)
    status = f"⚠️  {len(nulls)} column(s) have NULLs" if nulls else "✅ No NULLs"
    print(f"  {tbl_name:42s} {status}")
    for row in nulls:
        print(f"    └── {row['column']}: {row['null_count']:,} nulls ({row['null_pct']}%)")
        issues.append({
            "table": tbl_name,
            "check_type": "NULL values",
            "column": row["column"],
            "severity": row["severity"],
            "details": f"{row['null_count']:,} nulls ({row['null_pct']}% of {row['total_rows']:,} rows)",
        })

print("\n📋 2. BUSINESS RULE VIOLATIONS")
print("─" * 70)

def rule(label, n, table, col, msg):
    status = "✅ OK" if n == 0 else f"⚠️  {n:,} rows"
    print(f"  {label:55s} {status}")
    if n > 0:
        issues.append({
            "table": table,
            "check_type": "Business rule",
            "column": col,
            "severity": "HIGH",
            "details": msg(n),
        })

rules = business_rule_violations(claims, events, premiums)
rule("claims: date_of_loss > reported_date", rules["claims_loss_after_reported"],
     "bronze_claims_bordereau", "date_of_loss / reported_date",
     lambda n: f"{n:,} claims where loss date is after reported date")
rule("claims: incurred_amount < 0", rules["claims_negative_incurred"],
     "bronze_claims_bordereau", "incurred_amount",
     lambda n: f"{n:,} claims with negative incurred amount")
rule("claims: paid_to_date > incurred_amount", rules["claims_paid_exceeds_incurred"],
     "bronze_claims_bordereau", "paid_to_date",
     lambda n: f"{n:,} claims where amount paid exceeds amount incurred")
rule("premiums: policy_start_date >= policy_end_date", rules["premiums_bad_dates"],
     "bronze_premium_bordereau", "policy_start_date / policy_end_date",
     lambda n: f"{n:,} policies with start date on or after end date")
rule("premiums: sum_insured <= 0", rules["premiums_non_positive_sum_insured"],
     "bronze_premium_bordereau", "sum_insured",
     lambda n: f"{n:,} policies with zero or negative sum insured")
rule("premiums: annual_premium <= 0", rules["premiums_non_positive_annual_premium"],
     "bronze_premium_bordereau", "annual_premium",
     lambda n: f"{n:,} policies with zero or negative annual premium")
rule("cyclone_events: start_date > end_date", rules["events_start_after_end"],
     "bronze_cyclone_events", "start_date / end_date",
     lambda n: f"{n:,} events where start date is after end date")

print("\n📋 3. REFERENTIAL INTEGRITY")
print("─" * 70)

ri = referential_integrity_violations(claims, events, premiums, risk_zone)
rule("claims.policy_id → premium_bordereau (policy_id)", ri["orphan_claim_policies"],
     "bronze_claims_bordereau", "policy_id",
     lambda n: f"{n:,} claims reference a policy_id missing from premium_bordereau")
rule("claims.event_id → cyclone_events (non-null only)", ri["orphan_claim_events"],
     "bronze_claims_bordereau", "event_id",
     lambda n: f"{n:,} claims reference an event_id missing from cyclone_events")
rule("premium.postcode → risk_zone_lookup (postcode)", ri["orphan_premium_postcodes"],
     "bronze_premium_bordereau", "postcode",
     lambda n: f"{n:,} premiums reference a postcode missing from risk_zone_lookup")

print(f"\n{'═' * 70}")
print(f"  TOTAL ISSUES FOUND: {len(issues)}")
print(f"{'═' * 70}")

if issues:
    issues_df = spark.createDataFrame(issues).orderBy("severity", "table", "check_type")
    display(issues_df)
else:
    print("  ✅ All checks passed — no data quality issues detected.")

In [ ]:
from pyspark.sql import functions as F
from actuarial_claim_pipeline.data_quality import risk_zone_uniqueness

risk_zone = spark.table(f"{catalog}.{schema}.bronze_risk_zone_lookup")
uniq = risk_zone_uniqueness(risk_zone)

print(f"Total rows        : {uniq['total_rows']:,}")
print(f"Unique postcodes  : {uniq['unique_postcodes']:,}")

if uniq["is_unique"]:
    print("\n✅ PASS — Exactly one row per postcode. Lookup table is clean.")
else:
    print(
        f"\n⚠️  FAIL — {uniq['extra_rows']:,} extra row(s) detected "
        f"({uniq['total_rows']} rows vs {uniq['unique_postcodes']} unique postcodes)."
    )
    print("\nPostcodes with more than one row:")
    duplicated_postcodes = (
        risk_zone.groupBy("postcode", "region_name", "wind_risk_band")
        .agg(F.count("*").alias("row_count"))
        .filter(F.col("row_count") > 1)
        .orderBy(F.col("row_count").desc())
    )
    display(duplicated_postcodes)

    print("\nConflicting values for the same postcode (different region / risk band):")
    postcode_counts = (
        risk_zone.groupBy("postcode")
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct("region_name").alias("distinct_regions"),
            F.countDistinct("wind_risk_band").alias("distinct_risk_bands"),
        )
        .filter(F.col("row_count") > 1)
    )
    display(
        risk_zone.join(postcode_counts.select("postcode"), on="postcode", how="inner").orderBy("postcode")
    )